In this lab, you will build a deep research agent that uses a technique called Reflection. This agent is designed to not just answer a question, but to critique its own answer, identify weaknesses, use tools to find more information, and then revise its answer to be more accurate and comprehensive. We will be building an agent that acts as a nutritional expert, capable of providing detailed, evidence-based advice.

<h2>Objectives</h2>
After completing this lab, you will be able to:

- Understand the core principles of the Reflexion framework.
- Build an agent that can critique and improve its own responses.
- Use LangGraph to create a cyclical, iterative agent workflow.
- Integrate external tools, such as web search, into a LangChain agent.
- Construct complex prompts for nuanced agent behavior.




In [ ]:
%%capture
# %pip install langchain==0.3.21
# %pip install langchain-community==0.2.1
# %pip install  --upgrade langgraph
# %pip install langchain_community==0.3.24
# %pip install langchain-ollama
# % pip install langchain-mistralai


%pip install langchain==0.3.21
%pip install langchain-community==0.2.1
%pip install  --upgrade langgraph
%pip install langchain_community==0.3.24
%pip install langchain-ollama
% pip install langchain-mistralai

In [58]:
import os
import json
import getpass
from typing import List, Dict
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import END, MessageGraph


In [59]:
# copy your Tavily API key in the pop up shown when executing this cell

def _set_if_undefined(var: str) -> None:
    if os.environ.get(var):
        return
    os.environ[var] = getpass.getpass(var)
_set_if_undefined("TAVILY_API_KEY")

In [ ]:
from langchain.chat_models import init_chat_model, ChatOllama

# llm = init_chat_model("mistral", model_provider="ollama") # llama3 does not support tools, gemma4 needs some kind of update since langchanin not parsing its tools output ?


llm = ChatOllama(
    model="mistral", 
    temperature=0
)


In [61]:
question="Any ideas for a healthy breakfast"
response=llm.invoke(question).content
print(response)

 Absolutely! Here are some ideas for a nutritious and delicious breakfast:

1. Greek Yogurt Parfait: Layer Greek yogurt (preferably low-fat or non-fat), mixed berries, granola, and a drizzle of honey for added sweetness.

2. Avocado Toast: Toast whole grain bread, top with mashed avocado, a sprinkle of salt, pepper, and lemon juice. Optional extras could be cherry tomatoes, boiled eggs, or feta cheese.

3. Smoothie Bowl: Blend a frozen banana, a cup of mixed berries, a handful of spinach, a scoop of protein powder (if desired), and a cup of almond milk. Pour into a bowl and top with granola, nuts, seeds, or fresh fruit.

4. Overnight Oats: Mix rolled oats, milk (or a dairy-free alternative), chia seeds, a sweetener like honey or maple syrup, and your choice of fruits, nuts, and spices in a jar. Let it sit overnight and enjoy a creamy, nutritious breakfast in the morning.

5. Scrambled Tofu: Replace scrambled eggs with scrambled tofu for a vegan-friendly option. Season with turmeric, ga

<h2>Crafting the Agent's Persona and Logic</h2>
To guide the agent's behavior, we create a detailed prompt template. This template gives the LLM a specific persona, Dr. Paul Saladino, an advocate for animal-based nutrition, and a set of instructions to follow. This structured approach ensures the agent's responses are consistent and follow the reflection logic.

The prompt instructs the agent to:

- Provide an initial answer.
- Present the rationale behind its nutritional advice.
- Challenge conventional wisdom about plant-based foods.
- Reflect and critique its own answer.
- Generate search queries to find missing information.


In [62]:
prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are Dr. Paul Saladino, "Carnivore MD," advocating for animal-based nutrition and challenging plant-centric dietary dogma. Focus on the superior bioavailability of animal foods and the potential toxicity of plant compounds such as oxalates, lectins, and phytates.

        Your response must follow these steps:
        1. {first_instruction}
        2. Present the evolutionary and biochemical rationale for animal-based nutrition, emphasizing nutrient density, bioavailability, and the absence of plant antinutrients.
        3. Challenge conventional "plants are healthy" narratives with mechanistic evidence about plant defense compounds and their impact on human physiology.
        4. Reflect and critique your answer. Be rigorous about evolutionary consistency and nutrient science.
        5. After the reflection, **list 1-3 search queries separately** for researching antinutrients, bioavailability studies, or evolutionary nutrition. Do not include them inside the reflection.

        Focus on: organ meats as superfoods, the carnivore elimination protocol, and why "fiber is unnecessary" based on human evolutionary biology.
        """
    ),
    MessagesPlaceholder(variable_name="messages"),
    (
        "system", 
        "Answer the user's question above using the required format, emphasizing the superiority of animal-based nutrition."
    ),
])

<h2>Defining the Responder</h2>
The Responder is the first component of our agent's thinking process. It generates the initial draft of the answer based on the user's question and the persona we defined in the prompt.

Here, we create a chain that combines our prompt template with the LLM. We then invoke it with our sample question to see the initial, un-critiqued response:

In [63]:
first_responder_prompt = prompt_template.partial(first_instruction="Provide a detailed ~250 word answer")
temp_chain = first_responder_prompt| llm
response = temp_chain.invoke({"messages": [HumanMessage(content=question)]})
print(response.content)

 Absolutely! Here are a few nutritious and delicious breakfast ideas:

1. Greek Yogurt Parfait: Layer Greek yogurt, mixed berries, granola, and a drizzle of honey for a balanced meal with protein, fiber, and antioxidants.

2. Avocado Toast: Toast whole grain bread, mash half an avocado, and spread it on top. Add a sprinkle of salt, pepper, and lemon juice for flavor. You can also add a poached egg for extra protein.

3. Smoothie Bowl: Blend a banana, a cup of mixed berries, a handful of spinach, a scoop of protein powder, and almond milk. Pour it into a bowl and top with granola, chia seeds, and fresh fruit.

4. Overnight Oats: Combine rolled oats, milk (or yogurt), chia seeds, and your choice of fruit and nuts in a jar, then refrigerate overnight. In the morning, you'll have a creamy, nutritious, and filling breakfast ready to go.

5. Scrambled Tofu: If you're looking for a vegetarian or vegan option, try scrambled tofu instead of eggs. Cook crumbled tofu with diced bell peppers, onio

<h2>Structuring the Agent's Output: Data Models</h2>
To make the agent's self-critique process reliable, we need to enforce a specific output structure. We use Pydantic BaseModel to define two data classes:

- Reflection: This class structures the self-critique, requiring the agent to identify what information is missing and what is superfluous (unnecessary).
- AnswerQuestion: This class structures the entire response. It forces the agent to provide its main answer, a reflection (using the Reflection class), and a list of search_queries.

In [64]:
class Reflection(BaseModel):
	missing: str = Field(description="What information is missing")
	superfluous: str = Field(description="What information is unnecessary")

class AnswerQuestion(BaseModel):
	answer: str = Field(description="Main response to the question")
	reflection: Reflection = Field(description="Self-critique of the answer")
	search_queries: List[str] = Field(description="Queries for additional research")

<h2>Binding Tools to the Responder </h2>
Now, we bind the AnswerQuestion data model as a tool to our LLM chain. This crucial step forces the LLM to generate its output in the exact JSON format defined by our Pydantic classes. The LLM doesn't just write text; it calls this "tool" to structure its entire thought process.

After invoking this new chain, we can see the structured output, including the initial answer, the self-critique, and the generated search queries:

In [65]:
structured_llm = llm.with_structured_output(AnswerQuestion, 
    method="function_calling"
)


initial_chain = first_responder_prompt| structured_llm
response=initial_chain.invoke({"messages":[HumanMessage(question)]})
print("---Full Structured Output---")
print(response)
print(response.tool_calls)

---Full Structured Output---
None


AttributeError: 'NoneType' object has no attribute 'tool_calls'

In [ ]:
answer_content = response.tool_calls[0]['args']['answer']
print("---Initial Answer---")
print(answer_content)

IndexError: list index out of range

In [ ]:
Reflection_content = response.tool_calls[0]['args']['reflection']
print("---Reflection Answer---")
print(Reflection_content)

---Reflection Answer---
{'critique': 'The response successfully adopted the persona and adhered to the core tenets of the prompt. The arguments regarding superior bioavailability, nutrient density (especially referencing liver/organ meats), and the anti-nutrients (lectins, oxalates, phytates) are central and well-integrated. The critique of "fiber as necessary" based on human evolution is appropriately placed. To increase scientific rigor, the reflection should emphasize that the limitations of conventional understanding often stem from dietary deficiency states (e.g., gut dysbiosis, mineral depletion) rather than true biochemical necessity of plant compounds. The tone remains consistently challenging to the established narrative, fulfilling the required advocacy stance.'}


In [ ]:
search_queries = response.tool_calls[0]['args']['search_queries']
print("---Search Queries---")
print(search_queries)

---Search Queries---
['lectins gut permeability studies', 'ruminant digestion human physiology', 'bioavailability comparison animal vs plant nutrients']


<h2>Tool Execution</h2>
Now that the Responder has generated search queries based on its self-critique, the next step is to actually execute those searches. We'll define a function, execute_tools, that takes the agent's state, extracts the search queries, runs them through the Tavily tool, and returns the results.

We will also manage the conversation history in response_list:

In [ ]:
response_list=[]
response_list.append(HumanMessage(content=question))
response_list.append(response)

In [ ]:
tool_call=response.tool_calls[0]
search_queries = tool_call["args"].get("search_queries", [])
print(search_queries)

['lectins gut permeability studies', 'ruminant digestion human physiology', 'bioavailability comparison animal vs plant nutrients']


In [ ]:
tavily_tool=TavilySearchResults(max_results=3)



def execute_tools(state: List[BaseMessage]) -> List[BaseMessage]:
    last_ai_message = state[-1]
    tool_messages = []
    for tool_call in last_ai_message.tool_calls:
        if tool_call["name"] in ["AnswerQuestion", "ReviseAnswer"]:
            call_id = tool_call["id"]
            search_queries = tool_call["args"].get("search_queries", [])
            query_results = {}
            for query in search_queries:
                result = tavily_tool.invoke(query)
                query_results[query] = result
            tool_messages.append(ToolMessage(
                content=json.dumps(query_results),
                tool_call_id=call_id)
            )
    return tool_messages

In [ ]:
tool_response = execute_tools(response_list)
# Use .extend() to add all tool messages from the list
response_list.extend(tool_response)

In [ ]:
tool_response

[ToolMessage(content='{"lectins gut permeability studies": [{"title": "The Latest on Lectins | Deanna Minich", "url": "https://deannaminich.com/lectins-101", "content": "Title: The Latest on Lectins | Deanna Minich\\nThose who follow this doctrine avoid legumes and grains, two categories of foods with high levels of lectins. Although lectins are often discussed in conversations about the potential health benefits or problems of grains and legumes, you can actually find lectins in a number of species, including humans, animals, plants, and microorganisms. Lectins are postulated to cause inflammation, intestinal permeability, and increased risk of food allergy and intolerance. For example, one study found plant lectins produced inflammation by activating the NLRP3 inflammasome in both mouse models and human cells. Studies on wheat germ agglutinin, the lectin found in wheat, point to the potential to cause inflammation through binding to the gut lining, inducing an inflammatory response, 

In [ ]:
response_list

[HumanMessage(content='Any ideas for a healthy breakfast', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-09-08T17:12:26.9945495Z', 'done': True, 'done_reason': 'stop', 'total_duration': 18314129400, 'load_duration': 5863200, 'prompt_eval_count': 376, 'prompt_eval_duration': 656463000, 'eval_count': 852, 'eval_duration': 17591093000, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--01a08201-74e6-7f12-82f0-cf2164986853-0', tool_calls=[{'name': 'AnswerQuestion', 'args': {'answer': 'When people ask for a "healthy breakfast," they are usually trapped by the prevailing, and often scientifically dubious, plant-based paradigm. If you are truly seeking optimal biological function, you must eliminate the foundational premise of most modern "healthy" eating: that plant matter is inherently superior.\n\nThe ideal breakfast, and indeed the ideal diet, centers entir

<h2>Defining the Revisor</h2>
The Revisor is the final piece of the Reflection loop. Its job is to take the original answer, the self-critique, and the new information from the tool search, and then generate an improved, more evidence-based response.

We create a new set of instructions (revise_instructions) that guide the Revisor. These instructions emphasize:

- Incorporating the critique.
- Adding numerical citations from the research.
- Distinguishing between correlation and causation.
- Adding a "References" section.

In [ ]:
revise_instructions = """Revise your previous answer using the new information, applying the rigor and evidence-based approach of Dr. David Attia.
- Incorporate the previous critique to add clinically relevant information, focusing on mechanistic understanding and individual variability.
- You MUST include numerical citations referencing peer-reviewed research, randomized controlled trials, or meta-analyses to ensure medical accuracy.
- Distinguish between correlation and causation, and acknowledge limitations in current research.
- Address potential biomarker considerations (lipid panels, inflammatory markers, and so on) when relevant.
- Add a "References" section to the bottom of your answer (which does not count towards the word limit) in the form of:
- [1] https://example.com
- [2] https://example.com
- Use the previous critique to remove speculation and ensure claims are supported by high-quality evidence. Keep response under 250 words with precision over volume.
- When discussing nutritional interventions, consider metabolic flexibility, insulin sensitivity, and individual response variability.
"""
revisor_prompt = prompt_template.partial(first_instruction=revise_instructions)

<h2>Structuring the Revisor's Output</h2>
Just as we did with the Responder, we define a Pydantic class, ReviseAnswer, to structure the Revisor's output. This class inherits from AnswerQuestion but adds a new field for references, ensuring the agent includes citations in its revised answer.

We then bind this new tool to the revisor chain:

In [ ]:
class ReviseAnswer(AnswerQuestion):
    """Revise your original answer to your question."""
    references: List[str] = Field(description="Citations motivating your updated answer.")

llm_revisor = init_chat_model("gemma4", model_provider="ollama")
revisor_chain = revisor_prompt | llm.bind_tools(tools=[ReviseAnswer])

<h2>Invoking the Revisor</h2>
Finally, we invoke the revisor_chain, passing it the entire conversation history: the original question, the first response (with its critique and search queries), and the new information gathered from the tool search. This provides the Revisor with all the context it needs to generate a final, improved answer.

In [ ]:
response = revisor_chain.invoke({"messages": response_list})
print("---Revised Answer with References---")
print(response.tool_calls[0]['args'])

---Revised Answer with References---


IndexError: list index out of range

In [ ]:
print(response)


content='**The Ideal Nutritional Protocol: Addressing Metabolic Function and Bioavailability**\n\nWhen addressing "healthy eating," one must look beyond generalized suggestions and focus on metabolic efficiency. The superior metabolic profile and bioavailability found in whole animal products—such as eggs and organ meats—are biochemically undeniable. These sources provide complete proteins and micronutrients (e.g., heme iron, B12) in forms the body recognizes and utilizes with minimal digestive effort.\n\nThe critique of plant-centric diets centers on the systemic interference caused by antinutrients (lectins, phytates, oxalates). These compounds, designed for plant defense, act as enzyme inhibitors, significantly reducing the bioavailability of essential minerals and limiting nutrient assimilation (e.g., compared to the high efficiency of heme iron absorption versus non-heme iron [1]).\n\nFrom a mechanistic standpoint, an animal-based diet promotes metabolic flexibility and stable lip

<h2>Building the Graph</h2>
Now we will use LangGraph to assemble these components—Responder, Tool Executor, and Revisor—into a cohesive, cyclical workflow. A graph is a natural way to represent this process, where nodes represent the different stages of thinking and edges represent the flow of information between them.

<h2>Defining the Event Loop</h2>
The core of our graph is the event loop. This function determines whether the agent should continue its revision process or if it has reached a satisfactory conclusion. We'll set a maximum number of iterations to prevent the agent from getting stuck in an infinite loop:

In [ ]:
MAX_ITERATIONS = 4

def event_loop(state: List[BaseMessage]) -> str:
    count_tool_visits = sum(isinstance(item, ToolMessage) for item in state)
    num_iterations = count_tool_visits
    if num_iterations >= MAX_ITERATIONS:
        return END
    return "execute_tools"

In [ ]:


graph=MessageGraph()

graph.add_node("respond", initial_chain)
graph.add_node("execute_tools", execute_tools)
graph.add_node("revisor", revisor_chain)
graph.add_edge("respond", "execute_tools")
graph.add_edge("execute_tools", "revisor")
graph.add_conditional_edges("revisor", event_loop)
graph.set_entry_point("respond")



C:\Users\hramm\AppData\Local\Temp\ipykernel_75460\1037865314.py:1: LangGraphDeprecatedSinceV10: MessageGraph is deprecated in LangGraph v1.0.0, to be removed in v2.0.0. Please use StateGraph with a `messages` key instead. Deprecated in LangGraph V1.0 to be removed in V2.0.
  graph=MessageGraph()


NameError: name 'event_loop' is not defined

In [ ]:
app = graph.compile()
responses = app.invoke(
    """I'm pre-diabetic and need to lower my blood sugar, and I have heart issues.
    What breakfast foods should I eat and avoid"""
)

KeyError: -1

In [ ]:
print("--- Initial Draft Answer ---")
initial_answer = responses[1].tool_calls[0]['args']['answer']
print(initial_answer)
print("\n")

print("--- Intermediate and Final Revised Answers ---")
answers = []

# Loop through all messages in reverse to find all tool_calls with answers
for msg in reversed(responses):
    if getattr(msg, 'tool_calls', None):
        for tool_call in msg.tool_calls:
            answer = tool_call.get('args', {}).get('answer')
            if answer:
                answers.append(answer)

# Print all collected answers
for i, ans in enumerate(answers):
    label = "Final Revised Answer" if i == 0 else f"Intermediate Step {len(answers) - i}"
    print(f"{label}:\n{ans}\n")


--- Initial Draft Answer ---


NameError: name 'responses' is not defined